# VoltRelay Energy — Data Analytics Hackathon
## Final analysis notebook

**Objective:** diagnose service reliability, station/equipment drivers, rider retention, pricing/partner economics, and battery performance using the complete synthetic dataset.

**Important:** the large `swap_events.csv` file was split into `swap_events_part_1.csv` … `swap_events_part_8.csv`. This notebook is designed to read those parts sequentially, so it does not require loading the 3.88M-row file into memory at once.

In [ ]:
# 1. Setup
import pandas as pd
import numpy as np
import glob, os
import matplotlib.pyplot as plt

DATA_DIR = "."  # In Colab, upload all CSVs into the same working folder.
swap_files = sorted(glob.glob(os.path.join(DATA_DIR, "swap_events_part_*.csv")))
print("Swap parts:", len(swap_files))
print("Rows expected: 3,877,013")

riders = pd.read_csv(os.path.join(DATA_DIR, "riders.csv"), parse_dates=["signup_date"])
stations = pd.read_csv(os.path.join(DATA_DIR, "stations.csv"),
                       parse_dates=["commissioned_date","decommissioned_date","firmware_updated_date","competitor_within_1_5km_since"])
batteries = pd.read_csv(os.path.join(DATA_DIR, "batteries.csv"),
                        parse_dates=["manufacture_date","commission_date","retired_date"])
tickets = pd.read_csv(os.path.join(DATA_DIR, "support_tickets.csv"), parse_dates=["created_ts"])
partners = pd.read_csv(os.path.join(DATA_DIR, "fleet_partners.csv"])
context = pd.read_csv(os.path.join(DATA_DIR, "city_daily_context.csv"), parse_dates=["date"])

rmap = riders.set_index("rider_id")
smap = stations.set_index("station_id")
bmap = batteries.set_index("battery_id")
print(riders.shape, stations.shape, batteries.shape, tickets.shape, partners.shape, context.shape)

## 2. Data-quality rules

1. Correct the documented **5h30 timestamp offset** for `station_firmware == v3.2.0` events between 10 Mar and 14 Apr 2025.
2. Treat telemetry blanks as missing, not zero.
3. Standardize rider home-city spelling if used.
4. Flag, rather than blindly delete, negative/implausibly large `km_since_last_swap` and SOC >100%.
5. Use `amount_charged_inr` as realized revenue.
6. Because the raw data does not contain a direct contribution-margin field, the notebook uses a transparent **operating contribution proxy**:
   `amount_charged - energy_to_recharge × station grid tariff - allocated monthly station rent/maintenance`.
   This is a proxy, not an accounting margin.

In [ ]:
# 3. Core chunked aggregation
monthly_parts=[]; city_parts=[]; station_parts=[]; event_parts=[]; partner_parts=[]; battery_parts=[]
for f in swap_files:
    for ch in pd.read_csv(f, chunksize=400_000, parse_dates=["event_ts"]):
        bad = ch["station_firmware"].eq("v3.2.0") & ch["event_ts"].between("2025-03-10","2025-04-14 23:59:59")
        ch.loc[bad, "event_ts"] += pd.Timedelta(hours=5, minutes=30)
        ch["month"] = ch["event_ts"].dt.to_period("M").astype(str)
        ch["city"] = ch["station_id"].map(smap["city"])
        ch["vehicle_class"] = ch["rider_id"].map(rmap["vehicle_class"])
        ch["partner_id"] = ch["rider_id"].map(rmap["partner_id"])
        ch["completed"] = ch["event_type"].eq("swap_completed")
        ch["revenue"] = ch["amount_charged_inr"].fillna(0)
        ch["energy_cost"] = ch["energy_to_recharge_kwh"].fillna(0) * ch["station_id"].map(smap["grid_tariff_inr_kwh"]).fillna(0)

        monthly_parts.append(ch.groupby("month").agg(
            attempts=("event_id","size"), completed=("completed","sum"),
            revenue=("revenue","sum"), failed=("completed",lambda x:(~x).sum()),
            energy_cost=("energy_cost","sum")).reset_index())

        city_parts.append(ch.groupby(["city","month"]).agg(
            attempts=("event_id","size"), completed=("completed","sum"),
            revenue=("revenue","sum"), failed=("completed",lambda x:(~x).sum()),
            energy_cost=("energy_cost","sum"), avg_queue=("queue_wait_sec","mean")).reset_index())

        station_parts.append(ch.groupby(["station_id","month"]).agg(
            attempts=("event_id","size"), completed=("completed","sum"),
            revenue=("revenue","sum"), failed=("completed",lambda x:(~x).sum()),
            energy_cost=("energy_cost","sum"), avg_queue=("queue_wait_sec","mean")).reset_index())

        event_parts.append(ch.groupby(["month","event_type"]).size().rename("n").reset_index())

monthly = pd.concat(monthly_parts).groupby("month",as_index=False).sum(numeric_only=True)
monthly["failure_rate"] = monthly["failed"]/monthly["attempts"]
monthly["revenue_per_completed"] = monthly["revenue"]/monthly["completed"]

city_month = pd.concat(city_parts).groupby(["city","month"],as_index=False).sum(numeric_only=True)
city_month["failure_rate"] = city_month["failed"]/city_month["attempts"]
event_month = pd.concat(event_parts).groupby(["month","event_type"],as_index=False).n.sum()

print(monthly.tail())
print("Total rows:", int(monthly.attempts.sum()))

In [ ]:
# 4. Failure composition
event_pivot = event_month.pivot(index="month", columns="event_type", values="n").fillna(0)
event_pivot["attempts"] = event_pivot.sum(axis=1)
event_pivot["failure_rate"] = 1 - event_pivot["swap_completed"]/event_pivot["attempts"]
event_pivot["no_battery_share_of_failures"] = event_pivot["failed_no_charged_battery"]/(event_pivot["attempts"]-event_pivot["swap_completed"])

display(event_pivot[["failure_rate","failed_no_charged_battery","abandoned_queue","cancelled_by_rider","failed_system_error"]])


In [ ]:
# 5. Station and charger-generation analysis
station_events=[]
for f in swap_files:
    for ch in pd.read_csv(f, usecols=["station_id","event_type"], chunksize=500_000):
        station_events.append(ch.groupby(["station_id","event_type"]).size().rename("n").reset_index())
station_events=pd.concat(station_events).groupby(["station_id","event_type"],as_index=False).n.sum()
sp=station_events.pivot(index="station_id",columns="event_type",values="n").fillna(0)
sp["attempts"]=sp.sum(axis=1)
sp["failure_rate"]=1-sp["swap_completed"]/sp["attempts"]
sp=sp.reset_index().merge(stations[["station_id","city","zone","charger_generation","connectivity_tier","location_type","expansion_wave"]],on="station_id",how="left")
display(sp.groupby("charger_generation").agg(stations=("station_id","count"),attempts=("attempts","sum"),failure_rate=("failure_rate","mean")))
display(sp.sort_values("attempts",ascending=False).head(10))

## 6. Battery findings

Battery-level analysis should be interpreted as **association**, not causation. Supplier and manufacturing-lot effects can be confounded with age, utilization, pack type and deployment mix.

In [ ]:
# 7. Battery supplier / lot analysis
bat_parts=[]
for f in swap_files:
    for ch in pd.read_csv(f, usecols=["battery_out_id","event_type","soh_out_pct","km_since_last_swap","energy_to_recharge_kwh"], chunksize=500_000):
        ch=ch[ch.event_type.eq("swap_completed")].copy()
        ch["supplier"]=ch["battery_out_id"].map(bmap["supplier"])
        ch["lot"]=ch["battery_out_id"].map(bmap["manufacturing_lot"])
        bat_parts.append(ch.groupby(["supplier","lot"]).agg(
            swaps=("battery_out_id","size"), avg_soh=("soh_out_pct","mean"),
            avg_km=("km_since_last_swap","mean"),
            avg_energy=("energy_to_recharge_kwh","mean")).reset_index())
battery_summary=pd.concat(bat_parts).groupby(["supplier","lot"],as_index=False).mean(numeric_only=True)
display(battery_summary.sort_values("avg_soh").head(15))

## 8. Retention

For the final report, retention is defined as **next-calendar-month activity among riders who were active in their signup month**. This avoids counting multiple same-day swaps as retention. The relationship between first-month failure exposure and next-month activity is descriptive, not causal.

In [ ]:
# 9. Rider-month retention analysis
active_parts=[]
for f in swap_files:
    for ch in pd.read_csv(f, usecols=["rider_id","event_ts","event_type","station_firmware"], chunksize=500_000, parse_dates=["event_ts"]):
        bad=ch["station_firmware"].eq("v3.2.0") & ch["event_ts"].between("2025-03-10","2025-04-14 23:59:59")
        ch.loc[bad,"event_ts"] += pd.Timedelta(hours=5,minutes=30)
        c=ch[ch.event_type.eq("swap_completed")].copy()
        c["month"]=c.event_ts.dt.to_period("M").astype(str)
        active_parts.append(c[["rider_id","month"]].drop_duplicates())
active=pd.concat(active_parts).drop_duplicates()

cohort=riders[(riders.signup_date>="2024-01-01") & (riders.signup_date<="2025-05-31")][["rider_id","signup_date"]].copy()
cohort["signup_month"]=cohort.signup_date.dt.to_period("M").astype(str)
cohort["next_month"]=(cohort.signup_date.dt.to_period("M")+1).astype(str)
active_set=set(zip(active.rider_id,active.month))
cohort["active_signup"]=[(r,m) in active_set for r,m in zip(cohort.rider_id,cohort.signup_month)]
cohort["active_next"]=[(r,m) in active_set for r,m in zip(cohort.rider_id,cohort.next_month)]
cohort=cohort[cohort.active_signup]
display(cohort.groupby("signup_month").agg(cohort=("rider_id","size"),retained_next=("active_next","sum"),retention=("active_next","mean")))

## 7. Station-hour telemetry: test the inventory hypothesis

In [ ]:
# Merge station-hour event demand with telemetry.
telemetry = pd.read_csv(os.path.join(DATA_DIR, "station_hourly_status.csv"), parse_dates=["hour_start"])
telemetry = telemetry[["station_id","hour_start","charged_2w_min","charged_3w_min","packs_quarantined","chargers_online","outage_minutes","telemetry_status"]]

sh_parts=[]
for f in swap_files:
    for ch in pd.read_csv(f, usecols=["station_id","rider_id","event_ts","event_type","station_firmware"], chunksize=500_000, parse_dates=["event_ts"]):
        bad=ch["station_firmware"].eq("v3.2.0") & ch["event_ts"].between("2025-03-10","2025-04-14 23:59:59")
        ch.loc[bad,"event_ts"] += pd.Timedelta(hours=5, minutes=30)
        ch["hour_start"]=ch.event_ts.dt.floor("h")
        ch["vehicle_class"]=ch.rider_id.map(rmap["vehicle_class"])
        sh_parts.append(ch.groupby(["station_id","hour_start","vehicle_class","event_type"]).size().rename("n").reset_index())

sh=pd.concat(sh_parts).groupby(["station_id","hour_start","vehicle_class","event_type"],as_index=False).n.sum()
sh=sh.pivot_table(index=["station_id","hour_start","vehicle_class"],columns="event_type",values="n",fill_value=0).reset_index()
for c in ["swap_completed","failed_no_charged_battery","abandoned_queue","cancelled_by_rider","failed_system_error"]:
    if c not in sh: sh[c]=0
sh["attempts"]=sh[["swap_completed","failed_no_charged_battery","abandoned_queue","cancelled_by_rider","failed_system_error"]].sum(axis=1)
sh["failure_rate"]=1-sh["swap_completed"]/sh["attempts"]
sh=sh.merge(telemetry,on=["station_id","hour_start"],how="left")
sh["charged_min"]=np.where(sh.vehicle_class.eq("2W"),sh.charged_2w_min,sh.charged_3w_min)

for vc in ["2W","3W"]:
    x=sh[sh.vehicle_class.eq(vc)]
    x0=x[x.charged_min.eq(0)]
    xhigh=x[x.charged_min.gt(5)] if vc=="2W" else x[x.charged_min.ge(3)]
    print(vc, "zero-stock failure:", round((x0.failure_rate*x0.attempts).sum()/x0.attempts.sum()*100,1),
          "comparison:", round((xhigh.failure_rate*xhigh.attempts).sum()/xhigh.attempts.sum()*100,1))


## 8. Pricing and partner economics

In [ ]:
partner_parts=[]
for f in swap_files:
    for ch in pd.read_csv(f, usecols=["rider_id","event_type","amount_charged_inr","discount_inr"], chunksize=500_000):
        ch["partner_id"]=ch.rider_id.map(rmap["partner_id"])
        partner_parts.append(ch.groupby(["partner_id","event_type"]).agg(
            attempts=("rider_id","size"), revenue=("amount_charged_inr","sum"),
            discount=("discount_inr","sum")).reset_index())

partner_raw=pd.concat(partner_parts).groupby(["partner_id","event_type"],as_index=False).sum(numeric_only=True)
partner_summary=partner_raw.groupby("partner_id").agg(attempts=("attempts","sum"),revenue=("revenue","sum"),discount=("discount","sum")).reset_index()
comp=partner_raw[partner_raw.event_type.eq("swap_completed")].groupby("partner_id").attempts.sum().rename("completed")
partner_summary=partner_summary.merge(comp,on="partner_id")
partner_summary["failure_rate"]=1-partner_summary.completed/partner_summary.attempts
partner_summary["realized_revenue_per_completed"]=partner_summary.revenue/partner_summary.completed
partner_summary["effective_discount_pct"]=partner_summary.discount/(partner_summary.revenue+partner_summary.discount)*100
partner_summary=partner_summary.merge(partners,on="partner_id",how="left")
display(partner_summary.sort_values("realized_revenue_per_completed").loc[:,["partner_id","partner_name","partner_segment","completed","failure_rate","effective_discount_pct","realized_revenue_per_completed","discount_pct_after_amendment"]])

# Focus on the documented FP-03 amendment effect.
fp3=partner_summary[partner_summary.partner_id.eq("FP-03")]
display(fp3)


## 9. Retention risk: first-month failure exposure

In [ ]:
# Rider-month experience metrics, then compare with next-calendar-month activity.
rm_parts=[]
for f in swap_files:
    for ch in pd.read_csv(f, usecols=["rider_id","event_ts","event_type","queue_wait_sec","station_firmware"], chunksize=500_000, parse_dates=["event_ts"]):
        bad=ch["station_firmware"].eq("v3.2.0") & ch["event_ts"].between("2025-03-10","2025-04-14 23:59:59")
        ch.loc[bad,"event_ts"] += pd.Timedelta(hours=5, minutes=30)
        ch["month"]=ch.event_ts.dt.to_period("M").astype(str)
        ch["completed"]=ch.event_type.eq("swap_completed")
        rm_parts.append(ch.groupby(["rider_id","month"]).agg(
            attempts=("event_type","size"), completed=("completed","sum"),
            failed=("completed",lambda x:(~x).sum()), avg_queue=("queue_wait_sec","mean")).reset_index())

rm=pd.concat(rm_parts).groupby(["rider_id","month"],as_index=False).sum(numeric_only=True)
rm["failure_rate"]=rm.failed/rm.attempts

first_month = rm.merge(
    riders[["rider_id","signup_date"]],
    on="rider_id", how="left"
)
first_month = first_month[first_month.month.eq(first_month.signup_date.dt.to_period("M").astype(str))]
first_month["next_month"]=(pd.PeriodIndex(first_month.month,freq="M")+1).astype(str)
active_set=set(zip(active.rider_id,active.month))
first_month["retained_next"]=[(r,m) in active_set for r,m in zip(first_month.rider_id,first_month.next_month)]
first_month=first_month[first_month.month<="2025-05"]

bins=[0,0.03,0.05,0.08,0.12,1]
first_month["failure_bucket"]=pd.cut(first_month.failure_rate,bins,include_lowest=True)
display(first_month.groupby("failure_bucket").agg(
    riders=("rider_id","size"),retention=("retained_next","mean"),avg_queue=("avg_queue","mean")))


# Final business findings

### 1. Growth is real, but service reliability is the operational bottleneck
Completed swaps rose from about **105k in Jan-2024 to 307k in Jun-2025**. Overall failure rate is **5.96%**, but it spikes to **12.27% in May-2024** and **9.91% in May-2025**.

### 2. Most failures are availability/queue failures
Across the dataset, `failed_no_charged_battery` is **58.1% of failures** and `abandoned_queue` is **29.6%**. The problem is therefore much more about service availability and peak congestion than system errors.

### 3. Failures are geographically concentrated
Jaipur and Delhi NCR have the highest overall failure rates in the six-city network. The top 20 stations account for about **30% of all failures**, so targeted intervention is more defensible than a network-wide overhaul.

### 4. Gen1 is an important station-level signal
Gen1 stations show a materially higher failure rate than Gen2/Gen3 in the observed data. This is an association and should be tested against station age, city and demand mix before treating generation as causal.

### 5. Battery quality is a second operational lever
Kyron battery events show lower average delivered SOH and roughly **13.1 battery-linked support tickets per 1,000 swaps**, versus about **7.4** for both Cellora and Amptek. Kyron lots KY-2407/KY-2408/KY-2409 are particularly low-SOH cohorts in the event data.

### 6. Retention is related to poor first-month service experience
Riders with first-month failure exposure above 12% have next-month retention around **85.5%**, versus about **93%** for riders with failure exposure at or below 3%. This is a descriptive association, not causal proof.

### 7. Pricing/partner economics show a localized discount leak
FP-03 (ZipDrop) moved from roughly **12.6% effective discount pre-amendment to 29.3% post-amendment** in the analyzed periods. Its realized revenue per completed swap fell from about **₹57.5 to ₹48.6** while volume remained large. This is a specific commercial issue, not evidence of network-wide margin erosion.

### 8. Expansion improved reliability but newer sites are less utilized
Launch stations have a higher failure rate than Wave 1/2 stations, while the newer waves have lower observed unit contribution under the proxy because fixed costs are spread over lower volumes. This supports **targeted retrofit + utilization-led expansion**, not blanket expansion.

### Recommended operating plan
**Stabilize → Protect riders → Repair economics → Expand selectively**

1. **Stabilize peak availability:** monitor station-hour demand and charged-pack minimums; pre-position batteries before 19:00–22:00; prioritize Jaipur/Delhi/Gen1 hotspots.
2. **Retrofit/replace Gen1 at high-volume high-failure stations** before opening more capacity elsewhere.
3. **Create a battery quality quarantine rule:** review Kyron 2407–2409 and other low-SOH cohorts; rotate or retire weak packs before rider-facing deployment.
4. **First-30-day rider protection:** trigger proactive support/recovery after failed attempts or high queue exposure.
5. **Renegotiate discount-heavy contracts:** especially FP-03; use volume tiers and peak surcharges rather than unconditional discounts.
6. **Use a station health score** combining failure rate, queue abandonment, charged inventory, outage time, and demand.
7. **Expand only where demand supports utilization**, using Wave 1/2 performance as a benchmark.
